<img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png"  width='15%'>  


<h1> Demo: Building a Simple RAG with LangChain and Faiss</h1>
Developed by WeCloudData
<br></br>

---

## What You Need for RAG Implementation

Before building out a RAG system, it's essential to familiarize yourself with the tools that make this process possible.

Each tool plays a specific role, ensuring that the RAG system operates efficiently and effectively.

- **LLM**: At the heart of the system is the LLM, the core AI model responsible for generating human-like text responses.

- **Vector Store**: This is where the magic happens. The Vector Store is a dedicated storage system that houses embeddings and their corresponding textual data, ensuring quick and efficient retrieval.

- **Vector Store Retriever**: Think of this as the search engine of the system. The Vector Store Retriever fetches relevant documents by comparing vector similarities, ensuring that the most pertinent information is always at hand.

- **Embedder**: Before storing or retrieving data, we need to convert textual information into a format the system can understand. The Embedder takes on this role, transforming text into vector representations.

- **Prompt**: Every interaction starts with a user's query or statement. The Prompt captures this initial input, setting the stage for the retrieval and generation processes.

- **Document Loader**: With vast amounts of data to process, the Document Loader is essential. It imports and reads documents, preparing them for chunking and embedding.

- **Document Chunker**: To make the data more manageable and efficient for retrieval, the Document Chunker breaks documents into smaller, more digestible pieces.

- **User Input**: Last but not least, the User Input tool captures the query or statement provided by the end-user, initiating the entire RAG process.


## The RAG System and Its Subsystems

The primary goal of RAG is to provide LLMs with contextually relevant and factually accurate information, ensuring that the generated content meets the highest standards of quality and relevance.

To achieve this, the RAG system is divided into subsystems, each playing a crucial role in the overall process. The tools integral to the RAG system are not standalone entities; they interweave to form the subsystems that drive the RAG process.

Each tool fits within one of the following subsystems:

- 1) Index

- 2) Retrieval

- 3) Augment

These work together as an orchestrated flow that transforms a user's query into a contextually rich and accurate response.


In [1]:
!pip install langchain
!pip install langchain-community
!pip install langchain-openai
!pip install sentence-transformers
!pip install faiss-cpu
!pip install tiktoken
!pip install openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.0 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1:
      Successfully uninstalled langchain-core-1.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is inc

In [2]:
import os
import openai
from google.colab import userdata
userdata.get('OPENAI_API_KEY')

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
#openai.api_key = os.environ["OPENAI_API_KEY"]


# 1. Index System

**Purpose:** This subsystem is responsible for preparing and organizing the data for efficient retrieval.

Here are the steps of the Index system

**1) Load Documents (Document Loader)**: Imports and reads the vast amounts of data that the system will use.

**2) Chunk Documents (Document Chunker):** Breaks down the loaded documents into smaller, more manageable pieces to facilitate efficient retrieval.

**3) Embed Documents (Embedder):** Converts these textual chunks into vector representations, making them searchable within the system.

**4) Store Embeddings (Vector Store):** Safely stores the generated embeddings alongside their textual counterparts for future retrieval.


### Load documents

In [3]:
#!pip install langchain_community

In [4]:
from langchain_community.document_loaders import WebBaseLoader

yolo_nas_loader = WebBaseLoader("https://docs.ultralytics.com//").load()
decicoder_loader = WebBaseLoader("https://blogs.nvidia.com/blog/category/enterprise/deep-learning/").load()
yolo_newsletter_loader = WebBaseLoader("https://deeplearningdaily.substack.com/p/unleashing-the-power-of-yolo-nas").load()

In [5]:
# To show it was loaded correctly
str(yolo_nas_loader)

'[Document(metadata={\'source\': \'https://docs.ultralytics.com//\', \'title\': \'Home | Ultralytics Docs\', \'description\': \'Discover Ultralytics YOLO - the latest in real-time object detection and image segmentation. Learn about its features and maximize its potential in your projects.\', \'language\': \'en\'}, page_content=\'Home | Ultralytics DocsSkip to main contentUltralyticsSearch...HomeQuickstartHomeQuickstartUsageYOLO26 \\uf8ffüöÄModesTasksModelsCompareDatasetsSolutionsGuidesIntegrationsPlatformReferenceHelpLoadingPlease waitToggle SidebarDocsSwitch appsToggle themeEdit page on GitHub\\n\\n\\n\\n\\n\\n‰∏≠Êñá ¬∑\\nÌïúÍµ≠Ïñ¥ ¬∑\\nÊó•Êú¨Ë™û ¬∑\\n–†—É—Å—Å–∫–∏–π ¬∑\\nDeutsch ¬∑\\nFran√ßais ¬∑\\nEspa√±ol ¬∑\\nPortugu√™s ¬∑\\nT√ºrk√ße ¬∑\\nTi·∫øng Vi·ªát ¬∑\\nÿßŸÑÿπÿ±ÿ®Ÿäÿ©\\n\\n\\n\\n\\n\\n\\n\\n\\n\\n\\n\\n\\n\\n\\n\\nHome\\nIntroducing Ultralytics YOLO26, the latest version of the acclaimed real-time object detection and image segmentation model. YOLO26 is built on deep learning

In [6]:
# Review how many documents are there
len(yolo_nas_loader)

1

### Chunk documents
More about text splitters [here](https://python.langchain.com/docs/concepts/text_splitters/)

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Initialize the text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50,
    length_function = len
)

# Use the split_documents method
yolo_nas_chunks = text_splitter.split_documents(yolo_nas_loader)
decicoder_chunks = text_splitter.split_documents(decicoder_loader)
yolo_newsletter_chunks = text_splitter.split_documents(yolo_newsletter_loader)

In [9]:
print(f"Number of chunks for YOLO-NAS: {len(yolo_nas_chunks)}")
print(f"Number of chunks for Decicoder: {len(decicoder_chunks)}")
print(f"Number of chunks for YOLO Newsletter: {len(yolo_newsletter_chunks)}")

Number of chunks for YOLO-NAS: 29
Number of chunks for Decicoder: 14
Number of chunks for YOLO Newsletter: 18


### Create an index


In [ ]:
##### To delete the cache (for demonstration purposes only) ############
# import shutil
# import os
# #To delete cache to start from zero again
# cache_path = "./cache/"
# if os.path.exists(cache_path):
#     print("Deleting existing cache...")
#     shutil.rmtree(cache_path)

In [19]:
from langchain_openai import OpenAIEmbeddings
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_classic.storage import LocalFileStore
import time
import os

# 1. Set up the local cache store
store_path = "./cache/"
if not os.path.exists(store_path):
    os.makedirs(store_path)
    print(f"Created cache directory: {store_path}")

store = LocalFileStore(store_path)
print(f"Cache store initialized at {store_path}")
print("-" * 30)

# 2. Create the core embeddings model
print("Initializing the core OpenAI embedding model (text-embedding-3-small)...")
core_embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")

# 3. Create the cache-backed embedder
print("Initializing the CacheBackedEmbeddings wrapper...")
embedder = CacheBackedEmbeddings.from_bytes_store(
    core_embeddings_model,
    store,
    namespace=core_embeddings_model.model
)
print("Embedder is now configured to use the cache.")
print("-" * 30)

Created cache directory: ./cache/
Cache store initialized at ./cache/
------------------------------
Initializing the core OpenAI embedding model (text-embedding-3-small)...
Initializing the CacheBackedEmbeddings wrapper...
Embedder is now configured to use the cache.
------------------------------


/usr/local/lib/python3.12/dist-packages/langchain_classic/embeddings/cache.py:58: UserWarning: Using default key encoder: SHA-1 is *not* collision-resistant. While acceptable for most cache scenarios, a motivated attacker can craft two different payloads that map to the same cache key. If that risk matters in your environment, supply a stronger encoder (e.g. SHA-256 or BLAKE2) via the `key_encoder` argument. If you change the key encoder, consider also creating a new cache, to avoid (the potential for) collisions with existing keys.
  _warn_about_sha1_encoder()


In [20]:
# 4. Create the FAISS vector store from the first set of documents
# This is the first time we'll compute and store embeddings.
print(f"Adding {len(yolo_nas_chunks)} documents from 'YOLO-NAS' to the FAISS vector store.")
start_time = time.time()
vectorstore = FAISS.from_documents(yolo_nas_chunks, embedder)
end_time = time.time()
print(f"FAISS vector store created. Time taken: {end_time - start_time:.2f} seconds.")
print(f"The vector store now contains {vectorstore.index.ntotal} vectors.")
print("-" * 30)

# 5. Add the second set of documents
# Since these are new documents, the embeddings will be computed and cached.
print(f"Adding {len(decicoder_chunks)} documents from 'DeciCoder'...")
start_time = time.time()
vectorstore.add_documents(decicoder_chunks)
end_time = time.time()
print(f"Documents added. Time taken: {end_time - start_time:.2f} seconds.")
print(f"The vector store now contains a total of {vectorstore.index.ntotal} vectors.")
print("-" * 30)

# 6. Add the third set of documents
# Again, these are new documents, so they will be embedded and cached.
print(f"Adding {len(yolo_newsletter_chunks)} documents from 'YOLO Newsletter'...")
start_time = time.time()
vectorstore.add_documents(yolo_newsletter_chunks)
end_time = time.time()
print(f"Documents added. Time taken: {end_time - start_time:.2f} seconds.")
print(f"The vector store now contains a total of {vectorstore.index.ntotal} vectors.")
print("-" * 30)



Adding 29 documents from 'YOLO-NAS' to the FAISS vector store.
FAISS vector store created. Time taken: 2.50 seconds.
The vector store now contains 29 vectors.
------------------------------
Adding 14 documents from 'DeciCoder'...
Documents added. Time taken: 0.72 seconds.
The vector store now contains a total of 43 vectors.
------------------------------
Adding 18 documents from 'YOLO Newsletter'...
Documents added. Time taken: 1.19 seconds.
The vector store now contains a total of 61 vectors.
------------------------------


In [21]:
# 7. Instantiate a retriever for future use
# The retriever will be used to search the vector store for similar documents.
print("Converting the vector store into a retriever.")
retriever = vectorstore.as_retriever()
print("Retriever is ready!")
print("-" * 30)

# Optional check: Let's see if the cache works!
# If you run the script a second time, these next two lines
# will show a time close to 0 because the embeddings are loaded from the cache.
print("Re-adding the same documents to demonstrate caching...")
start_time = time.time()
vectorstore.add_documents(yolo_nas_chunks) # This should be very fast!
end_time = time.time()
print(f"Time taken to re-add cached documents: {end_time - start_time:.4f} seconds.")
print("-" * 30)

print("Process complete. You can now use the 'retriever' object for similarity searches.")

Converting the vector store into a retriever.
Retriever is ready!
------------------------------
Re-adding the same documents to demonstrate caching...
Time taken to re-add cached documents: 0.0230 seconds.
------------------------------
Process complete. You can now use the 'retriever' object for similarity searches.


# 2. Retrieval System

**Purpose:** As the name suggests, this subsystem fetches the most relevant information based on the user's query.

Here are the steps in the Retrieval system

**1) Obtain User Query (User Input):** Captures the user's question or statement.

**2) Embed User Query (Embedder):** Transforms the user's query into a vector format, similar to the indexed documents.

**3) Vector Search (Vector Store Retriever):** Searches the Vector Store for documents with embeddings that closely match the embedded user query.

**4) Return Relevant Documents:** The system then returns the top matches, ensuring that the most pertinent information is always provided.


In [22]:
from langchain_openai import ChatOpenAI
from langchain_classic.chains import RetrievalQA
from langchain_classic.callbacks import StdOutCallbackHandler

In [23]:
handler = StdOutCallbackHandler()

# Using 'model' is the current convention instead of 'model_name'
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0.0,
    streaming=False,
    callbacks=[handler],
)

In [24]:
# this is the entire retrieval system
qa_with_sources_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    callbacks=[handler],
    return_source_documents=True
)

# 3. Augment System

**Purpose:** This subsystem enhances the LLM's input prompt with the retrieved context, ensuring that the model has all the necessary information to generate a comprehensive response.

**1) Create Initial Prompt (Prompt):** Starts with the original user query or statement.

**2) Augment Prompt with Retrieved Context:** Merges the initial prompt with the context retrieved from the Vector Store, creating an enriched input for the LLM.

**3) Send Augmented Prompt to LLM:** The enhanced prompt is then fed to the LLM.

**4) Receive LLM's Response:** After processing the augmented prompt, the LLM generates and returns its response.


In [25]:
# This is the entire augment system!
response = qa_with_sources_chain({"query":"What is YoloNAS and what is the latest version?"})

/tmp/ipykernel_6555/1107296175.py:2: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  response = qa_with_sources_chain({"query":"What is YoloNAS and what is the latest version?"})




> Entering new RetrievalQA chain...

> Finished chain.


Look at the entire response  

In [26]:
response

{'query': 'What is YoloNAS and what is the latest version?',
 'result': 'YOLO-NAS is a family of object detection architectures that feature a novel quantization-friendly basic block, designed to improve performance and efficiency. It includes small, medium, and large versions optimized for various use cases. The development of YOLO-NAS involved perfecting the training regime to fully realize the potential of these innovative architectures.\n\nThe latest version mentioned is YOLO26, which continues the tradition of architectural improvements and innovative techniques. YOLO26 incorporates the latest advancements in computer vision research, including end-to-end NMS-free inference and optimized edge deployment for real-world applications.',
 'source_documents': [Document(id='f146fdcd-7230-4b29-bd80-987d18e977f7', metadata={'source': 'https://docs.ultralytics.com//', 'title': 'Home | Ultralytics Docs', 'description': 'Discover Ultralytics YOLO - the latest in real-time object detection an

If you want just the response

In [27]:
print(response['result'])

YOLO-NAS is a family of object detection architectures that feature a novel quantization-friendly basic block, designed to improve performance and efficiency. It includes small, medium, and large versions optimized for various use cases. The development of YOLO-NAS involved perfecting the training regime to fully realize the potential of these innovative architectures.

The latest version mentioned is YOLO26, which continues the tradition of architectural improvements and innovative techniques. YOLO26 incorporates the latest advancements in computer vision research, including end-to-end NMS-free inference and optimized edge deployment for real-world applications.


And you can get the source like so:

In [28]:
print(response['source_documents'])

[Document(id='f146fdcd-7230-4b29-bd80-987d18e977f7', metadata={'source': 'https://docs.ultralytics.com//', 'title': 'Home | Ultralytics Docs', 'description': 'Discover Ultralytics YOLO - the latest in real-time object detection and image segmentation. Learn about its features and maximize its potential in your projects.', 'language': 'en'}, page_content='With each new version, YOLO has introduced architectural improvements and innovative techniques that have enhanced performance across various metrics. YOLO26 continues this tradition by incorporating the latest advancements in computer vision research, featuring end-to-end NMS-free inference and optimized edge deployment for real-world applications.\nFAQ\nWhat is Ultralytics YOLO and how does it improve object detection?'), Document(id='8d201203-62fa-47ed-bfcb-d09bdde7eea8', metadata={'source': 'https://docs.ultralytics.com//', 'title': 'Home | Ultralytics Docs', 'description': 'Discover Ultralytics YOLO - the latest in real-time objec

In [29]:
response = qa_with_sources_chain({"query":"What is the latest new on deep learning"})



> Entering new RetrievalQA chain...

> Finished chain.


In [30]:
print(response['result'])

The latest news in deep learning includes the launch of OpenAI's GPT-5.2 model in December 2025, described as the most capable model series yet for professional knowledge work. Additionally, there are advancements in object detection with the introduction of YOLO-NAS, a novel architecture setting new state-of-the-art standards in computer vision. NVIDIA continues to play a significant role in powering AI developments, including partnerships and infrastructure support for autonomous AI agents and self-improving AI systems.


In [31]:
response = qa_with_sources_chain({"query":"Write a blog about Deci and how it used NAS to generate YOLO-NAS and DeciCoder"})



> Entering new RetrievalQA chain...

> Finished chain.


In [32]:
print(response['result'])

Title: Deci’s Breakthrough: How Neural Architecture Search Powered YOLO-NAS and DeciCoder

In the rapidly evolving world of artificial intelligence and computer vision, innovation often hinges on the ability to design models that are not only accurate but also efficient and scalable. Deci, a leading AI company, has made significant strides in this domain by leveraging Neural Architecture Search (NAS) to create two groundbreaking models: YOLO-NAS and DeciCoder. These models exemplify the power of automated architecture design in pushing the boundaries of performance and efficiency.

## What is Neural Architecture Search (NAS)?

Neural Architecture Search is an automated method for designing neural network architectures. Instead of manually crafting model structures, NAS algorithms explore a vast space of possible architectures to find the optimal design that balances accuracy, speed, and resource consumption. This approach accelerates innovation by discovering architectures that might b

In [33]:
print(response['source_documents'])

[Document(id='1d0d1e50-2bd1-41ed-9d6c-ac89b1abe162', metadata={'source': 'https://deeplearningdaily.substack.com/p/unleashing-the-power-of-yolo-nas', 'title': 'Unleashing the Power of YOLO-NAS: A New Era in Object Detection and Computer Vision', 'description': 'The Future of Computer Vision is Here', 'language': 'en'}, page_content='in the technical details of YOLO-NAS? Read the technical blog post.Read the Technical Blog PostPerfecting the Training RegimeThe team devised an advanced and innovative training scheme incorporating several elements, each carefully crafted to extract maximum performance from the model:Pretraining on the extensive Object365 dataset.Employing pseudo-labelled data to enhance learning.Applying knowledge distillation from a pre-trained teacher model to improve performance.After completing the'), Document(id='33c2ce55-4e1c-44a5-b695-9c6084b67069', metadata={'source': 'https://deeplearningdaily.substack.com/p/unleashing-the-power-of-yolo-nas', 'title': 'Unleashing

In [34]:
response = qa_with_sources_chain({"query":"How many version are there in YOLO"})



> Entering new RetrievalQA chain...

> Finished chain.


In [35]:
print(response['result'])

There are multiple versions of YOLO released over the years, including:

- YOLOv2 (2016)
- YOLOv3 (2018)
- YOLOv4 (2020)
- YOLO26 (a more recent version with advanced features like end-to-end NMS-free inference and optimized edge deployment)

These versions represent major milestones, each introducing architectural improvements and innovative techniques to enhance object detection performance.
